In [63]:
# import important libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from  sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV

In [64]:
cleaned_customer_transaction_data = pd.read_csv(r'../dataset/artifacts/Cleaned_Data.csv')

Threshold-based features were created based on key behavioural and risk indicators identified during exploratory analysis. These include low KYC verification levels location mismatches between the registered country and ip address and high-risk transaction characteristics such as high transaction amounts, elevated Ip risk score, and low device trust scores. These features were identified as high risk threshold due to their significantly elevated fraud rates. These thresholds were converted into binary risk indicators to enhance the model's ability to detect suspicious transactions

In [65]:
# creating a  threshold based feature from the following risk_signal
cleaned_customer_transaction_data['timestamp'] = pd.to_datetime(cleaned_customer_transaction_data['timestamp'])
cleaned_customer_transaction_data['hour'] = cleaned_customer_transaction_data['timestamp'].dt.hour
cleaned_customer_transaction_data['late_night_hours'] = ((cleaned_customer_transaction_data['hour'] >= 3) & (cleaned_customer_transaction_data['hour'] <= 7)).astype(int)
cleaned_customer_transaction_data['amount_high'] = (cleaned_customer_transaction_data['amount_usd'] >= 1000).astype(int)
cleaned_customer_transaction_data['high_ip_risk'] = (cleaned_customer_transaction_data['ip_risk_score'] >0.8).astype(int)
cleaned_customer_transaction_data['low_device_trust'] = (cleaned_customer_transaction_data['device_trust_score'] <0.5).astype(int)
cleaned_customer_transaction_data['new_account'] = ((cleaned_customer_transaction_data['account_age_days'] >=30)&(cleaned_customer_transaction_data['account_age_days'] < 90)).astype(int) 
cleaned_customer_transaction_data['very_new_account'] = (cleaned_customer_transaction_data['account_age_days'] < 30).astype(int)
cleaned_customer_transaction_data['velocity_spike'] = (cleaned_customer_transaction_data['txn_velocity_1h'] >= 3).astype(int)


high_risk_signal_features = cleaned_customer_transaction_data[['late_night_hours', 'amount_high', 'high_ip_risk', 'low_device_trust', 'new_account', 'velocity_spike']]
high_risk_signal_features.head()

,late_night_hours,amount_high,high_ip_risk,low_device_trust,new_account,velocity_spike
0,0,0,0,0,0,0
1,0,0,0,1,0,0
2,0,0,0,0,0,0
3,0,0,0,0,0,0
4,0,0,0,0,0,0


In [66]:
# checking the features in our dataset 
list(cleaned_customer_transaction_data.columns)

['transaction_id',
 'customer_id',
 'timestamp',
 'home_country',
 'source_currency',
 'dest_currency',
 'channel',
 'amount_src',
 'amount_usd',
 'fee',
 'exchange_rate_src_to_dest',
 'device_id',
 'new_device',
 'ip_address',
 'ip_country',
 'location_mismatch',
 'ip_risk_score',
 'kyc_tier',
 'account_age_days',
 'device_trust_score',
 'chargeback_history_count',
 'risk_score_internal',
 'txn_velocity_1h',
 'txn_velocity_24h',
 'corridor_risk',
 'is_fraud',
 'hour',
 'late_night_hours',
 'amount_high',
 'high_ip_risk',
 'low_device_trust',
 'new_account',
 'very_new_account',
 'velocity_spike']

FEATURE SELECTION

In [67]:
# Dropping all temporary buckets columns
#cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop(['device_trust_score'], axis=1)
# also i will be dropping all identifiers columns (ids)
cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop(['transaction_id', 'customer_id', 'device_id', 'ip_address'], axis=1)
# Dropping some irrelevant variables
cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop(['chargeback_history_count', 'exchange_rate_src_to_dest'], axis=1)

Defining Categorical Features

In [68]:
categorical_features = cleaned_customer_transaction_data.select_dtypes(include=['object', 'bool']).columns
categorical_features

C:\Users\pc\AppData\Local\Temp\ipykernel_13580\2235495287.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = cleaned_customer_transaction_data.select_dtypes(include=['object', 'bool']).columns


Index(['home_country', 'source_currency', 'dest_currency', 'channel',
       'new_device', 'ip_country', 'location_mismatch', 'kyc_tier'],
      dtype='str')

In [69]:
# Defining numerical features 
numerical_features = cleaned_customer_transaction_data.select_dtypes(include=['int', float]).columns.drop('is_fraud')
numerical_features

Index(['amount_src', 'amount_usd', 'fee', 'ip_risk_score', 'account_age_days',
       'device_trust_score', 'risk_score_internal', 'txn_velocity_1h',
       'txn_velocity_24h', 'corridor_risk', 'hour', 'late_night_hours',
       'amount_high', 'high_ip_risk', 'low_device_trust', 'new_account',
       'very_new_account', 'velocity_spike'],
      dtype='str')

In [70]:
print(f"categorical_features: {len(categorical_features)}")
print(f"numerical_features: {len(numerical_features)}")
print(f"Dataset: {cleaned_customer_transaction_data.shape}")


categorical_features: 8
numerical_features: 18
Dataset: (10804, 28)


In [71]:
cleaned_customer_transaction_data.columns

Index(['timestamp', 'home_country', 'source_currency', 'dest_currency',
       'channel', 'amount_src', 'amount_usd', 'fee', 'new_device',
       'ip_country', 'location_mismatch', 'ip_risk_score', 'kyc_tier',
       'account_age_days', 'device_trust_score', 'risk_score_internal',
       'txn_velocity_1h', 'txn_velocity_24h', 'corridor_risk', 'is_fraud',
       'hour', 'late_night_hours', 'amount_high', 'high_ip_risk',
       'low_device_trust', 'new_account', 'very_new_account',
       'velocity_spike'],
      dtype='str')

In [72]:
cleaned_customer_transaction_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10804 entries, 0 to 10803
Data columns (total 28 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   timestamp            10804 non-null  datetime64[us, UTC]
 1   home_country         10804 non-null  str                
 2   source_currency      10804 non-null  str                
 3   dest_currency        10804 non-null  str                
 4   channel              10804 non-null  str                
 5   amount_src           10804 non-null  float64            
 6   amount_usd           10804 non-null  float64            
 7   fee                  10804 non-null  float64            
 8   new_device           10804 non-null  bool               
 9   ip_country           10804 non-null  str                
 10  location_mismatch    10804 non-null  bool               
 11  ip_risk_score        10804 non-null  float64            
 12  kyc_tier             10804 no

Convert to CSV files

In [73]:
cleaned_customer_transaction_data.to_csv("../dataset/artifacts/Engineered_Data.csv", index=False)